# Retrieval

In [ ]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

persist_directory = "../data/chroma/"
embedding = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")
vectordb = Chroma(persist_directory=persist_directory, embedding_function=embedding)
print(vectordb._collection.count())

## Addressing diversity: Maximum Marginal Relevance (MMR)

In [ ]:
question = "ما هي شروط فسخ عقد الايجار؟"
docs_ss = vectordb.similarity_search(question, k=3)
docs_mmr = vectordb.max_marginal_relevance_search(question, k=3)

print("--- plain similarity search ---")
for d in docs_ss:
    print(d.page_content[:100], "\n")

print("--- MMR ---")
for d in docs_mmr:
    print(d.page_content[:100], "\n")

## Addressing specificity: metadata filtering

In [ ]:
docs = vectordb.similarity_search(
    question,
    k=3,
    filter={"source": "lloc"},  # legislation only
)
for d in docs:
    print(d.metadata)

## Self-query retriever

In [ ]:
from langchain_groq import ChatGroq
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever
from langchain_classic.chains.query_constructor.base import AttributeInfo

metadata_field_info = [
    AttributeInfo(
        name="source",
        description="Which of the three sources this chunk is from: 'lloc' (legislation), 'sjc' (Court of Cassation judgments), or 'ccb' (Constitutional Court rulings).",
        type="string",
    ),
    AttributeInfo(
        name="doc_id",
        description="The law code or case/appeal number this chunk belongs to.",
        type="string",
    ),
]

llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)

document_content_description = "Bahraini legal text: legislation articles and court judgments"
retriever = SelfQueryRetriever.from_llm(
    llm,
    vectordb,
    document_content_description,
    metadata_field_info,
    verbose=True,
)

In [ ]:
docs = retriever.invoke(question)
for d in docs:
    print(d.metadata)

## Other retrieval types

In [ ]:
from langchain_community.retrievers import TFIDFRetriever

all_texts = vectordb.get()["documents"]
tfidf_retriever = TFIDFRetriever.from_texts(all_texts)

docs_tfidf = tfidf_retriever.invoke(question)
docs_tfidf[0].page_content[:200] if docs_tfidf else "no results"